# Appliance Energy Forecasting

Google Colab-ready case study covering preprocessing, EDA, stationarity, benchmarks, SARIMAX, feature-based regression, optional Chronos, common metrics, diagnostics and report-question interpretations.

Run every cell in order with **Runtime → Run all**. The default AIC grid is bounded for practical execution. Set `FULL_AIC_GRID = True` in the configuration cell for the lecturer-required exhaustive non-seasonal grid; it may take several hours.

## 0. INSTALLATION AND IMPORTS

In [ ]:
import os
import sys
import subprocess
import importlib.util
import warnings
from pathlib import Path

REQUIRED = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "sklearn": "scikit-learn",
    "statsmodels": "statsmodels",
}
missing = [package for module, package in REQUIRED.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

import itertools
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller, kpss

warnings.filterwarnings("ignore")
np.random.seed(42)

## 1. CONFIGURATION

In [ ]:
DATA_URL = (
    "https://archive.ics.uci.edu/ml/machine-learning-databases/"
    "00374/energydata_complete.csv"
)
TARGET = "Appliances"
DAILY_PERIOD = 24
WEEKLY_PERIOD = 168
TEST_STEPS = 14 * 24
FORECAST_HORIZON = 24
RANDOM_STATE = 42

# False gives a practical demonstration grid. True searches p,q=0..6 and d=0..2.
FULL_AIC_GRID = False

# AIC selection window. The selected model is refitted to the complete training set.
SEARCH_WINDOW = 30 * 24

# Set True only if Colab can download the official Chronos weights.
RUN_CHRONOS = False

ROOT = Path("/content/appliance_energy_project") if Path("/content").exists() else Path("appliance_energy_project")
DATA_DIR = ROOT / "data"
OUTPUT_DIR = ROOT / "outputs"
FIGURE_DIR = OUTPUT_DIR / "figures"
METRICS_DIR = OUTPUT_DIR / "metrics"
FORECAST_DIR = OUTPUT_DIR / "forecasts"
DIAGNOSTIC_DIR = OUTPUT_DIR / "diagnostics"
for directory in [DATA_DIR, FIGURE_DIR, METRICS_DIR, FORECAST_DIR, DIAGNOSTIC_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

## 2. DATA DOWNLOAD AND PREPARATION

In [ ]:
def load_and_prepare_data(url=DATA_URL):
    """Download, validate, and resample the 10-minute data to hourly means."""
    print("Downloading UCI Appliances Energy Prediction dataset...")
    raw = pd.read_csv(url, parse_dates=["date"])
    raw = raw.set_index("date").sort_index()

    print(f"Raw shape: {raw.shape}")
    print(f"Raw period: {raw.index.min()} to {raw.index.max()}")
    print(f"Duplicate timestamps: {raw.index.duplicated().sum()}")
    print(f"Missing target observations: {raw[TARGET].isna().sum()}")

    if raw.index.has_duplicates:
        raw = raw.groupby(level=0).mean(numeric_only=True)

    raw = raw.apply(pd.to_numeric, errors="coerce")
    if raw[TARGET].isna().any():
        raise ValueError("The raw target contains missing or non-numeric values.")

    hourly = raw.resample("h").mean()
    hourly = hourly.interpolate(method="time", limit_area="inside")

    if hourly[TARGET].isna().any():
        raise ValueError("Missing target values remain after resampling.")

    hourly.to_csv(DATA_DIR / "appliance_hourly.csv")
    print(f"Hourly shape: {hourly.shape}")
    print(f"Total missing hourly values: {hourly.isna().sum().sum()}")
    return hourly


data = load_and_prepare_data()

## 3. EXPLORATORY DATA ANALYSIS

In [ ]:
def make_eda_plots(df):
    fig, axes = plt.subplots(3, 1, figsize=(15, 11))
    df[TARGET].plot(ax=axes[0], linewidth=0.6, color="#275D8C")
    axes[0].set_title("Hourly appliance energy use")
    axes[0].set_ylabel("Appliances")

    df[TARGET].tail(14 * 24).plot(ax=axes[1], linewidth=1.1, color="#275D8C")
    axes[1].set_title("Final 14 days")
    axes[1].set_ylabel("Appliances")

    daily_profile = df.assign(hour=df.index.hour).groupby("hour")[TARGET].mean()
    daily_profile.plot(ax=axes[2], marker="o", color="#C65D21")
    axes[2].set_title("Mean hour-of-day profile")
    axes[2].set_xlabel("Hour")
    axes[2].set_ylabel("Mean Appliances")

    for axis in axes:
        axis.grid(alpha=0.2)
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / "eda_overview.png", dpi=180, bbox_inches="tight")
    plt.show()


make_eda_plots(data)

print("\nTarget summary statistics:")
display(data[TARGET].describe()) if "display" in globals() else print(data[TARGET].describe())

print("\nMean correlations with Appliances:")
correlations = data.corr(numeric_only=True)[TARGET].sort_values(ascending=False)
display(correlations.head(12)) if "display" in globals() else print(correlations.head(12))

## 4. STATIONARITY TESTS

In [ ]:
def stationarity_tests(y):
    rows = []
    variants = {
        "level": y,
        "first_difference": y.diff(),
        "daily_difference": y.diff(DAILY_PERIOD),
    }
    for name, series in variants.items():
        series = series.dropna()
        adf_result = adfuller(series, autolag="AIC")
        kpss_result = kpss(series, regression="c", nlags="auto")
        rows.append({
            "transform": name,
            "ADF statistic": adf_result[0],
            "ADF p-value": adf_result[1],
            "KPSS statistic": kpss_result[0],
            "KPSS p-value": kpss_result[1],
        })
    return pd.DataFrame(rows)


stationarity = stationarity_tests(data[TARGET])
stationarity.to_csv(DIAGNOSTIC_DIR / "stationarity_tests.csv", index=False)
print("\nStationarity tests:")
display(stationarity) if "display" in globals() else print(stationarity)

## 5. TRAIN/TEST SPLIT AND METRICS

In [ ]:
y = data[TARGET]
train = y.iloc[:-TEST_STEPS]
test = y.iloc[-TEST_STEPS:]
print(f"\nTraining period: {train.index.min()} to {train.index.max()} ({len(train)} hours)")
print(f"Test period: {test.index.min()} to {test.index.max()} ({len(test)} hours)")
print(f"Operational forecast horizon: {FORECAST_HORIZON} hours")


def evaluate_forecast(name, actual, forecast, training, seasonality=24):
    actual = np.asarray(actual, dtype=float)
    forecast = np.asarray(forecast, dtype=float)
    training = np.asarray(training, dtype=float)
    scale = np.abs(training[seasonality:] - training[:-seasonality]).mean()
    denominator = (np.abs(actual) + np.abs(forecast)) / 2
    smape = np.divide(
        np.abs(actual - forecast), denominator,
        out=np.zeros_like(actual), where=denominator != 0,
    ).mean() * 100
    return {
        "model": name,
        "MAE": mean_absolute_error(actual, forecast),
        "RMSE": mean_squared_error(actual, forecast) ** 0.5,
        "MASE": np.abs(actual - forecast).mean() / scale,
        "Bias": np.mean(forecast - actual),
        "sMAPE_pct": smape,
    }

## 6. BENCHMARK FORECASTS

In [ ]:
def seasonal_naive(training, horizon, period):
    return np.resize(training.iloc[-period:].to_numpy(), horizon)


def benchmark_forecasts(training, forecast_index):
    horizon = len(forecast_index)
    steps = np.arange(1, horizon + 1)
    drift_slope = (training.iloc[-1] - training.iloc[0]) / (len(training) - 1)
    return pd.DataFrame({
        "mean": np.repeat(training.mean(), horizon),
        "naive": np.repeat(training.iloc[-1], horizon),
        "seasonal_naive_daily": seasonal_naive(training, horizon, DAILY_PERIOD),
        "seasonal_naive_weekly": seasonal_naive(training, horizon, WEEKLY_PERIOD),
        "drift": training.iloc[-1] + steps * drift_slope,
    }, index=forecast_index)


forecasts = benchmark_forecasts(train, test.index)

## 7. SARIMAX AIC SEARCH, FIT, FORECAST, AND DIAGNOSTICS

In [ ]:
def sarimax_aic_search(series, full_grid=False):
    """
    Search SARIMAX parameters using AIC.

    full_grid=True searches p,q=0..6 and d=0..2 as required by the brief.
    Seasonal P,D,Q are 0..1, with daily seasonal period 24.
    """
    p_values = range(7) if full_grid else range(3)
    d_values = range(3) if full_grid else range(2)
    q_values = range(7) if full_grid else range(3)
    seasonal_values = list(itertools.product(range(2), range(2), range(2)))
    combinations = list(itertools.product(p_values, d_values, q_values))
    total = len(combinations) * len(seasonal_values)
    print(f"\nTesting up to {total} SARIMAX parameter combinations...")

    rows = []
    completed = 0
    for p, d, q in combinations:
        for P, D, Q in seasonal_values:
            completed += 1
            try:
                model = SARIMAX(
                    series,
                    order=(p, d, q),
                    seasonal_order=(P, D, Q, DAILY_PERIOD),
                    enforce_stationarity=False,
                    enforce_invertibility=False,
                )
                fit = model.fit(disp=False, maxiter=80)
                rows.append({
                    "p": p, "d": d, "q": q,
                    "P": P, "D": D, "Q": Q,
                    "AIC": fit.aic,
                })
            except Exception:
                pass
            if completed % 25 == 0:
                print(f"Completed {completed}/{total}")

    if not rows:
        raise RuntimeError("Every SARIMAX candidate failed.")
    return pd.DataFrame(rows).sort_values("AIC").reset_index(drop=True)


search_series = train.tail(SEARCH_WINDOW)
aic_results = sarimax_aic_search(search_series, full_grid=FULL_AIC_GRID)
aic_results.to_csv(DIAGNOSTIC_DIR / "sarimax_aic_grid.csv", index=False)
print("\nBest AIC candidates:")
display(aic_results.head(10)) if "display" in globals() else print(aic_results.head(10))

best = aic_results.iloc[0]
best_order = tuple(int(best[x]) for x in ["p", "d", "q"])
best_seasonal_order = tuple(int(best[x]) for x in ["P", "D", "Q"]) + (DAILY_PERIOD,)
print(f"Selected order: {best_order}; seasonal order: {best_seasonal_order}")

sarimax_model = SARIMAX(
    train,
    order=best_order,
    seasonal_order=best_seasonal_order,
    enforce_stationarity=False,
    enforce_invertibility=False,
)
sarimax_fit = sarimax_model.fit(disp=False)
sarimax_result = sarimax_fit.get_forecast(steps=len(test))
forecasts["sarimax"] = sarimax_result.predicted_mean.to_numpy()
sarimax_ci = sarimax_result.conf_int(alpha=0.05)
sarimax_ci.index = test.index
sarimax_ci.to_csv(FORECAST_DIR / "sarimax_confidence_intervals.csv")

sarimax_residuals = test - forecasts["sarimax"]
ljung_box = acorr_ljungbox(sarimax_residuals, lags=[24, 48], return_df=True)
ljung_box.to_csv(DIAGNOSTIC_DIR / "sarimax_ljung_box.csv")
print("\nSARIMAX Ljung-Box residual tests:")
display(ljung_box) if "display" in globals() else print(ljung_box)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(sarimax_residuals, bins=30, color="#275D8C", alpha=0.85)
axes[0].set_title("SARIMAX residual distribution")
plot_acf(sarimax_residuals, lags=72, ax=axes[1])
plt.tight_layout()
plt.savefig(FIGURE_DIR / "sarimax_residuals.png", dpi=180, bbox_inches="tight")
plt.show()

## 8. LEAK-SAFE FEATURE ENGINEERING

In [ ]:
SENSOR_WEATHER_COLUMNS = [
    "T1", "RH_1", "T2", "RH_2", "T3", "RH_3", "T4", "RH_4",
    "T5", "RH_5", "T6", "RH_6", "T7", "RH_7", "T8", "RH_8",
    "T9", "RH_9", "T_out", "Press_mm_hg", "RH_out", "Windspeed",
    "Visibility", "Tdewpoint",
]


def create_features(df, include_realised_covariates=False):
    """Create time, lag, and shifted rolling features without target leakage."""
    output = pd.DataFrame(index=df.index)
    hour = df.index.hour
    day_of_week = df.index.dayofweek

    output["hour_sin"] = np.sin(2 * np.pi * hour / 24)
    output["hour_cos"] = np.cos(2 * np.pi * hour / 24)
    output["dow_sin"] = np.sin(2 * np.pi * day_of_week / 7)
    output["dow_cos"] = np.cos(2 * np.pi * day_of_week / 7)
    output["is_weekend"] = (day_of_week >= 5).astype(int)

    for lag in [1, 2, 3, 6, 12, 24, 48, 168]:
        output[f"lag_{lag}"] = df[TARGET].shift(lag)

    shifted_target = df[TARGET].shift(1)
    for window in [3, 6, 12, 24, 168]:
        output[f"roll_mean_{window}"] = shifted_target.rolling(window).mean()
        output[f"roll_std_{window}"] = shifted_target.rolling(window).std()

    if include_realised_covariates:
        available = [column for column in SENSOR_WEATHER_COLUMNS if column in df.columns]
        output[available] = df[available]

    output[TARGET] = df[TARGET]
    return output.dropna()


def train_feature_model(df, split_time, include_realised_covariates=False):
    table = create_features(df, include_realised_covariates)
    training_table = table.loc[table.index < split_time]
    testing_table = table.loc[table.index >= split_time]

    X_train = training_table.drop(columns=TARGET)
    y_train = training_table[TARGET]
    X_test = testing_table.drop(columns=TARGET)

    model = HistGradientBoostingRegressor(
        max_iter=350,
        learning_rate=0.04,
        max_leaf_nodes=31,
        l2_regularization=1.0,
        random_state=RANDOM_STATE,
    )
    model.fit(X_train, y_train)
    prediction = pd.Series(model.predict(X_test), index=X_test.index)
    return prediction, model, list(X_train.columns)


split_time = test.index[0]
operational_prediction, operational_model, operational_features = train_feature_model(
    data, split_time, include_realised_covariates=False
)
conditional_prediction, conditional_model, conditional_features = train_feature_model(
    data, split_time, include_realised_covariates=True
)
forecasts["feature_model_operational"] = operational_prediction.reindex(test.index)
forecasts["feature_model_conditional"] = conditional_prediction.reindex(test.index)

print(f"\nOperational feature count: {len(operational_features)}")
print(f"Conditional feature count: {len(conditional_features)}")
print("Note: realised future sensor/weather values make the second model conditional.")

## 9. OPTIONAL CHRONOS FOUNDATION MODEL

In [ ]:
def run_chronos(training, prediction_length):
    """Run the real Chronos-Bolt model; never substitutes a benchmark."""
    packages = ["chronos-forecasting", "torch", "socksio"]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])
    import torch
    from chronos import BaseChronosPipeline

    pipeline = BaseChronosPipeline.from_pretrained(
        "amazon/chronos-bolt-small", device_map="cpu"
    )
    context = torch.tensor(training.to_numpy(), dtype=torch.float32)
    quantiles, mean = pipeline.predict_quantiles(
        context,
        prediction_length=prediction_length,
        quantile_levels=[0.1, 0.5, 0.9],
    )
    return mean.detach().cpu().numpy().reshape(-1)


foundation_status = "Chronos was not requested. Set RUN_CHRONOS=True to run it."
if RUN_CHRONOS:
    try:
        forecasts["chronos_bolt"] = run_chronos(train, len(test))
        foundation_status = "Chronos-Bolt completed successfully."
    except Exception as error:
        foundation_status = f"Chronos could not run: {type(error).__name__}: {error}"
print("\nFoundation-model status:", foundation_status)
(DIAGNOSTIC_DIR / "foundation_status.txt").write_text(foundation_status)

## 10. MODEL EVALUATION

In [ ]:
forecast_output = forecasts.copy()
forecast_output.insert(0, "actual", test)
forecast_output.to_csv(FORECAST_DIR / "all_forecasts.csv")

metric_rows = []
for model_name in forecasts.columns:
    metric_rows.append(evaluate_forecast(
        model_name, test, forecasts[model_name], train, DAILY_PERIOD
    ))
metrics = pd.DataFrame(metric_rows).sort_values("RMSE").reset_index(drop=True)
metrics.to_csv(METRICS_DIR / "model_comparison.csv", index=False)
print("\nFINAL MODEL COMPARISON")
display(metrics.round(3)) if "display" in globals() else print(metrics.round(3))

## 11. FORECAST AND MODEL-COMPARISON PLOTS

In [ ]:
fig, axis = plt.subplots(figsize=(16, 7))
train.tail(WEEKLY_PERIOD).plot(axis=axis, label="Training history", color="#777777")
test.plot(ax=axis, label="Actual", color="black", linewidth=2.2)
for column in forecasts.columns:
    forecasts[column].plot(ax=axis, label=column, linewidth=1, alpha=0.8)
axis.set_title("Forecast comparison: final 14-day test period")
axis.set_ylabel("Appliances")
axis.legend(ncol=2, fontsize=8)
axis.grid(alpha=0.2)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "forecast_comparison.png", dpi=180, bbox_inches="tight")
plt.show()

sorted_metrics = metrics.sort_values("RMSE", ascending=True)
fig, axis = plt.subplots(figsize=(11, 5))
axis.barh(sorted_metrics["model"], sorted_metrics["RMSE"], color="#275D8C")
axis.set_xlabel("RMSE")
axis.set_title("Model accuracy: lower RMSE is better")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "model_rmse.png", dpi=180, bbox_inches="tight")
plt.show()

## 12. AUTOMATIC INTERPRETATION OF THE SIX REPORT QUESTIONS

In [ ]:
requested_benchmarks = [
    "naive", "seasonal_naive_daily", "seasonal_naive_weekly", "drift"
]
strongest_benchmark = (
    metrics[metrics["model"].isin(requested_benchmarks)]
    .sort_values("RMSE").iloc[0]
)
sarimax_metrics = metrics.loc[metrics["model"] == "sarimax"].iloc[0]
operational_metrics = metrics.loc[
    metrics["model"] == "feature_model_operational"
].iloc[0]

benchmark_gain = 100 * (
    strongest_benchmark["RMSE"] - sarimax_metrics["RMSE"]
) / strongest_benchmark["RMSE"]
feature_gain = 100 * (
    strongest_benchmark["RMSE"] - operational_metrics["RMSE"]
) / strongest_benchmark["RMSE"]

print("\nANSWERS TO THE SIX REQUIRED QUESTIONS")
print(f"1. Strongest requested benchmark: {strongest_benchmark['model']} "
      f"(RMSE={strongest_benchmark['RMSE']:.2f}).")
print(f"2. SARIMAX RMSE={sarimax_metrics['RMSE']:.2f}, an improvement of "
      f"{benchmark_gain:.1f}% over that benchmark. Inspect residual tests before "
      "claiming the dependence is fully captured.")
print(f"3. The operational feature model RMSE={operational_metrics['RMSE']:.2f}, "
      f"an improvement of {feature_gain:.1f}% over the strongest requested benchmark.")
if "chronos_bolt" in metrics["model"].values:
    chronos_metrics = metrics.loc[metrics["model"] == "chronos_bolt"].iloc[0]
    print(f"4. Chronos-Bolt RMSE={chronos_metrics['RMSE']:.2f}; compare this value "
          "with the feature model and its additional computational cost.")
else:
    print("4. Chronos was not completed, so no foundation-model performance claim "
          "should be made. Do not rename a seasonal benchmark as Chronos.")
print("5. Calendar variables and historical lags are known at forecast origin. "
      "Realised future indoor sensors and weather create a conditional forecast.")
print("6. Recommend the operational feature model if its advantage remains stable "
      "under rolling-origin evaluation; keep weekly seasonal naive as a fallback.")

summary = {
    "data_rows": len(data),
    "data_start": str(data.index.min()),
    "data_end": str(data.index.max()),
    "test_rows": len(test),
    "best_model": metrics.iloc[0]["model"],
    "best_rmse": float(metrics.iloc[0]["RMSE"]),
    "best_sarimax_order": best_order,
    "best_sarimax_seasonal_order": best_seasonal_order,
    "foundation_status": foundation_status,
}
(DIAGNOSTIC_DIR / "run_summary.json").write_text(json.dumps(summary, indent=2))

print(f"\nAll outputs saved in: {OUTPUT_DIR.resolve()}")
print("Finished successfully.")